In [23]:
import numpy as np

import gplately
import gplately.tools as tools
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader as shpreader
import netCDF4
import warnings
from scipy import ndimage
import glob, os
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import matplotlib.gridspec as gridspec
#from slabdip import SlabDipper
import matplotlib
import pandas as pd

import glob, os
import matplotlib.pyplot as plt
from plate_model_manager import PlateModelManager

from scipy.spatial import cKDTree

reconstruction_times = np.arange(0,1800,1)
extent_global = [-180,180,-90,90]

# Import 1.8Ga_model_optimised_mantle_ref_frame_20240725

In [24]:
use_local_files = True

if use_local_files:
    # Method 1: manually point to files
    #input_directory = 'shirmard25'
    #input_directory = r'./shirmard25'  # Raw string to avoid issues with backslashes
    input_directory = './shirmard25'

    # Load both rotation files
    rotation_filenames = glob.glob(os.path.join(input_directory, '*.rot')) # Include both rot files
    rotation_model = pygplates.RotationModel(rotation_filenames)

    # Load static polygons
    static_polygons = pygplates.FeatureCollection(os.path.join(input_directory, 'static_polygons.gpmlz'))
    COBs = os.path.join(input_directory, 'COBfile_1800_0.gpml')  # Use updated COB file

    # Load topology files and combine them into a single FeatureCollection
    topology_filenames = [
        os.path.join(input_directory, '250-0_plate_boundaries.gpml'),
        os.path.join(input_directory, '410-250_plate_boundaries.gpml'),
        os.path.join(input_directory, '1800-1000_plate_boundaries.gpml'),
        os.path.join(input_directory, '1000-410_plate-boundaries.gpml'),  # Corrected file name
        os.path.join(input_directory, '1000-410-Convergence.gpml'),
        os.path.join(input_directory, '1000-410-Divergence.gpml'),
        os.path.join(input_directory, '1000-410-Transforms.gpml'),
        os.path.join(input_directory, 'TopologyBuildingBlocks.gpml'),
    ]

    # Create an empty list to hold all the features
    all_topology_features = []

    # Load each file and append its features to the list
    for topology_filename in topology_filenames:
        fc = pygplates.FeatureCollection(topology_filename)
        all_topology_features.extend(fc)

    # Create a single FeatureCollection from the combined features
    topology_features = pygplates.FeatureCollection(all_topology_features)

    # Optionally load other geological features if needed
    continents = pygplates.FeatureCollection(os.path.join(input_directory, 'shapes_continents.gpmlz'))
    COBs = pygplates.FeatureCollection(os.path.join(input_directory, 'COBfile_1800_0.gpml'))
    shapes_coasts = pygplates.FeatureCollection(os.path.join(input_directory, 'shapes_coasts.gpmlz'))

    # Create PlateReconstruction model
    model = gplately.PlateReconstruction(rotation_model=rotation_model, 
                                         topology_features=topology_features, 
                                         static_polygons=static_polygons)

    # Now you can proceed with your reconstruction and plotting...


# Load Hoggard data

In [25]:
# shp

In [26]:
# Read the shapefile into a GeoDataFrame
gdf = gpd.read_file('./Inputs/Deposits_with_craton_parameters.shp')  # Replace with the actual path to your shapefile

# Rename the columns as requested
gdf = gdf.rename(columns={
    'TM2': 'weight',
    'Lat_': 'present_lat',
    'Lon_': 'present_lon',
    'Age': 'age (Ma)',
    'Type' : 'Type'
})

# Dictionary to hold DataFrames for each type
metal_dict = dict()

# List of commodities (not used in the filtering, just for reference)
commodities = ['Cu (Mt)', 'Pb (Mt)', 'Zn (Mt)', 'Ni (Mt)']

# Get unique types from the 'Type' column
types = gdf['Type'].unique()

# Filter data by type
for type_name in types:
    df = gdf[gdf['Type'] == type_name]
    
    # Ensure that the 'age (Ma)' column is present and filter data
    if 'age (Ma)' in df.columns:
        df = df[df['age (Ma)'].notna()]
        df = df[df['age (Ma)'] > 0]  # Remove rows where age = 0
        df = df[df['age (Ma)'] <= 1800]
        
        if df.shape[0] > 0:
            metal_dict[type_name] = df

# Process the filtered data
pts_dict = dict()
for i, t in enumerate(types):
    df = metal_dict[t]
    
    # Extract Lon/Lat from the geometry column and create points for gplately
    pts_dict[t] = gplately.Points(model, df['geometry'].x, df['geometry'].y)

# You can now proceed with the rest of your code (e.g., plotting symbols)
symbols = ['o', 'v', 's', '*', 'd', '^', 'P']*2


<!-- ### Create random continental locations

This is not really being used currently. We have to think more on how to compare our statistical correlations to a random baseline. -->

# Load craton database

In [27]:
reconstruction_time_map = 50

In [28]:
### Load craton database
Craton_HS_filename = './Inputs/merged_segmented_craton.shp'
Craton_HS_features = pygplates.FeatureCollection(Craton_HS_filename)
reconstructed_Craton_HS_features = model.reconstruct(Craton_HS_features, reconstruction_time_map)
#deposits = '/data/Craton_HS_2B_PLATEID1/Craton_Deposits_PLATEID1.gpml'

In [29]:
from scipy.spatial import cKDTree

def KD_dist(lons0, lats0, lons1, lats1, k=1):
    """ Function to efficiently query nearest-neighbour distances """
    x0, y0, z0 = gplately.tools.lonlat2xyz(lons0, lats0)
    x1, y1, z1 = gplately.tools.lonlat2xyz(lons1, lats1)
    
    tree = cKDTree(np.c_[x0, y0, z0])
    d, idx = tree.query(np.c_[x1, y1, z1], k=k)
    return d*gplately.EARTH_RADIUS, idx

In [30]:
# Initialize unique_times as a set to collect unique ages
unique_times = set()

# Loop over each type in types
for i, type_name in enumerate(types):
    df = metal_dict[type_name]
    
    # Collect unique ages from each type and update the set
    unique_times = unique_times.union(set(df['age (Ma)'].to_numpy(dtype=int)))

    # # Add new columns to the dataframe with default values
    # # Distance attributes
    # df['distance_to_craton'] = 0.0
    # df['distance_to_random'] = 0.0
    # df['distance_craton_to_trench'] = 0.0

    # # Rift attributes
    # df['craton_age'] = 0.0
    # df['craton_length'] = 0.0
    # df['craton_azimuth'] = 0.0
    # df['craton_velocity'] = 0.0
    # df['craton_velocity_azimuth'] = 0.0

    # # Subduction zone attributes
    # df['trench_convergence_velocity'] = 0.0
    # df['trench_advance'] = 0.0
    # df['trench_azimuth'] = 0.0
    # df['trench_craton_obliquity'] = 0.0

    # Store the updated dataframe back in the metal_dict
    metal_dict[type_name] = df


In [31]:
# change the year (g)

# Merge (1,2,3,4,5) and Extract All Features

In [32]:
# Read the shapefile into a GeoDataFrame
gdf = gpd.read_file('./Inputs/Deposits_with_craton_parameters.shp')

# Rename the columns
gdf = gdf.rename(columns={
    'TM2': 'weight',
    'Lat_': 'present_lat',
    'Lon_': 'present_lon',
    'Age': 'age (Ma)',
    'Type': 'Type'
})

# Dictionary to hold DataFrames for each type
metal_dict = dict()

# Get unique types from the 'Type' column
types = gdf['Type'].unique()

# Filter data by type
for type_name in types:
    df = gdf[gdf['Type'] == type_name].copy()  # Use .copy() to avoid warnings
    
    # Ensure that the 'age (Ma)' column is present and filter data
    if 'age (Ma)' in df.columns:
        df = df[df['age (Ma)'].notna()]
        df = df[df['age (Ma)'] > 0]  # Remove rows where Age = 0
        df = df[df['age (Ma)'] <= 1800]
        
        if df.shape[0] > 0:
            # Reset index to ensure continuous indexing from 0
            df = df.reset_index(drop=True)
            metal_dict[type_name] = df

# Process the filtered data
pts_dict = dict()
for i, t in enumerate(types):
    df = metal_dict[t]
    
    # Extract Lon/Lat from the geometry column and create points for gplately
    pts_dict[t] = gplately.Points(model, df['geometry'].x, df['geometry'].y)

# Configuration parameters
a = 20  # Years for median calculations
g = 20  # Years for gradient calculations

# Main processing loop
for t, time in enumerate(unique_times):
    # Reconstruct craton features for the current time
    reconstructed_craton_features = model.reconstruct(Craton_HS_features, time)

    # Tessellate subduction zones for the current time
    subduction_data = model.tessellate_subduction_zones(time, np.deg2rad(0.5), ignore_warnings=True)
    subduction_lon = subduction_data[:, 0]
    subduction_lat = subduction_data[:, 1]
    subduction_vel = subduction_data[:, 2] * 1e-2
    subduction_angle = subduction_data[:, 3]
    subduction_norm = subduction_data[:, 7]
    subduction_pid_sub = subduction_data[:, 8]
    subduction_pid_over = subduction_data[:, 9]
    subduction_length = np.deg2rad(subduction_data[:, 6]) * gplately.EARTH_RADIUS * 1e3  # in meters
    subduction_convergence = np.fabs(subduction_data[:, 2]) * 1e-2 * np.cos(np.radians(subduction_data[:, 3]))
    subduction_migration = np.fabs(subduction_data[:, 4]) * 1e-2 * np.cos(np.radians(subduction_data[:, 5]))

    # Convert lon/lat to xyz coordinates for subduction zones
    sx, sy, sz = gplately.tools.lonlat2xyz(subduction_lon, subduction_lat)
    tree_sz = cKDTree(np.c_[sx, sy, sz])  # Create KDTree for nearest neighbor search

    # Calculate craton centroids and create KDTree
    craton_centroids = np.empty((len(reconstructed_craton_features), 2))
    for f, feature in enumerate(reconstructed_craton_features):
        geometry = feature.get_reconstructed_geometry()
        craton_centroids[f, :] = geometry.get_centroid().to_lat_lon()

    # Convert craton centroids to xyz coordinates for KDTree search
    cx, cy, cz = gplately.tools.lonlat2xyz(craton_centroids[:, 1], craton_centroids[:, 0])
    tree_craton = cKDTree(np.c_[cx, cy, cz])  # Create KDTree for nearest craton search

    # Iterate over each mineral type
    for type_name in types:
        df = metal_dict[type_name]
        deposit_age = df['age (Ma)'].to_numpy(dtype=int)
        mask_deposit_ages = deposit_age == time
        index_deposit_age = df.index[mask_deposit_ages].tolist()

        if len(index_deposit_age) == 0:
            continue

        gpts = pts_dict[type_name]
        rlons, rlats = gpts.reconstruct(time, return_array=True)

        # Apply strict validation for latitude and longitude values
        valid_mask = (rlats >= -90) & (rlats <= 90) & (rlons >= -180) & (rlons <= 180)

        # Filter valid latitudes and longitudes
        rlons = rlons[valid_mask]
        rlats = rlats[valid_mask]

        if len(rlats) == 0 or len(rlons) == 0:
            print(f"No valid lat/lon points for {type_name} at time {time}")
            continue

        # Calculate deposit velocities at time of formation
        deposit_velocity_components = model.get_point_velocities(rlons, rlats, time)
        deposit_velocity_magnitude = np.hypot(deposit_velocity_components[:, 0], deposit_velocity_components[:, 1])

        # Get the subset of valid deposits (after filtering)
        rlons_masked = rlons[mask_deposit_ages[valid_mask]]
        rlats_masked = rlats[mask_deposit_ages[valid_mask]]
        
        if len(rlons_masked) == 0 or len(rlats_masked) == 0:
            print(f"No valid masked deposits for {type_name} at time {time}")
            continue

        # Find closest craton to the deposits
        distance_to_craton, neighbours = KD_dist(
            craton_centroids[:, 1], craton_centroids[:, 0], rlons_masked, rlats_masked
        )

        # ==================== INSTANTANEOUS VALUES (at time of formation) ====================
        # Create a mapping of valid indices
        valid_indices = np.where(valid_mask)[0]
        masked_valid_indices = valid_indices[mask_deposit_ages[valid_mask]]
        
        for loop_idx, (d2r, neighbour, df_index) in enumerate(zip(distance_to_craton, neighbours, index_deposit_age)):
            feature = reconstructed_craton_features[neighbour]
            geometry = feature.get_reconstructed_geometry()

            # Get craton properties
            craton_length = geometry.get_arc_length() * 6371.0  # Convert arc length to kilometers
            craton_coordinates = geometry.to_lat_lon_array()

            # Calculate craton azimuth
            yx_distances = craton_coordinates[1:] - craton_coordinates[:-1]
            craton_angles = np.rad2deg(np.arctan2(yx_distances[:, 1], yx_distances[:, 0]))
            craton_angles[craton_angles < 0] += 180

            # Calculate craton velocities and azimuth
            craton_velocity_components = model.get_point_velocities(craton_coordinates[:, 1], craton_coordinates[:, 0], time)
            craton_velocity_magnitude = np.hypot(craton_velocity_components[:, 0], craton_velocity_components[:, 1])
            craton_velocity_angle = np.rad2deg(np.arctan2(craton_velocity_components[:, 0], craton_velocity_components[:, 1]))
            craton_velocity_angle[craton_velocity_angle < 0] += 180

            # Find nearest subduction zone segment
            rx, ry, rz = gplately.tools.lonlat2xyz(craton_coordinates[:, 1], craton_coordinates[:, 0])
            d2s, sz_neighbours = tree_sz.query(np.c_[rx, ry, rz])
            d2s_mean = d2s.mean() * 6371  # Convert distance to kilometers

            # Calculate subduction properties for the nearest neighbour
            sz_convergence = subduction_convergence[sz_neighbours].mean()
            sz_migration = subduction_migration[sz_neighbours].mean()
            sz_norm = subduction_norm[sz_neighbours].copy()

            # Normalize sz_norm to be between 0 and 180
            sz_norm[sz_norm > 180] -= 180

            # Calculate the smallest angle difference (obliquity) between the craton and subduction zone
            sz_obliquity = abs(sz_norm.mean() - craton_angles.mean()) % 180

            # Limit the obliquity to a maximum of 90 degrees (as obliquity should be between 0 and 90)
            if sz_obliquity > 90:
                sz_obliquity = 180 - sz_obliquity

            # Get the correct deposit velocity and position for this specific deposit
            if loop_idx < len(masked_valid_indices):
                actual_idx = masked_valid_indices[loop_idx]
                deposit_vel = deposit_velocity_magnitude[actual_idx] if actual_idx < len(deposit_velocity_magnitude) else deposit_velocity_magnitude.mean()
                rlat_val = rlats[actual_idx] if actual_idx < len(rlats) else rlats_masked[loop_idx]
                rlon_val = rlons[actual_idx] if actual_idx < len(rlons) else rlons_masked[loop_idx]
            else:
                deposit_vel = deposit_velocity_magnitude.mean()
                rlat_val = rlats_masked[loop_idx] if loop_idx < len(rlats_masked) else rlats_masked[0]
                rlon_val = rlons_masked[loop_idx] if loop_idx < len(rlons_masked) else rlons_masked[0]

            # Update the DataFrame with instantaneous values using .at for single value assignment
            df.at[df_index, 'cr_ve'] = craton_velocity_magnitude.mean()
            df.at[df_index, 'dp_ve'] = deposit_vel
            df.at[df_index, 'ds_cr_tr'] = d2s_mean
            df.at[df_index, 'sb_ve'] = sz_convergence
            df.at[df_index, 'tr_adv'] = sz_migration
            df.at[df_index, 'tr_cr_an'] = sz_obliquity
            df.at[df_index, 'lat'] = rlat_val
            df.at[df_index, 'lon'] = rlon_val

        # ==================== MEDIAN VALUES (over 'a' years) ====================
        # Initialize lists for median calculations
        deposit_velocity_values = []
        craton_velocity_values = []
        craton_velocity_azimuth_values = []
        trench_craton_obliquity_values = []
        trench_convergence_values = []
        trench_advance_values = []

        # Loop for the 'a' years prior to the deposit's formation age
        for year_offset in range(a):
            avg_time = time - year_offset
            gpts = pts_dict[type_name]
            rlons_temp, rlats_temp = gpts.reconstruct(avg_time, return_array=True)

            valid_mask_temp = (rlats_temp >= -90) & (rlats_temp <= 90) & (rlons_temp >= -180) & (rlons_temp <= 180)
            rlons_temp = rlons_temp[valid_mask_temp]
            rlats_temp = rlats_temp[valid_mask_temp]

            if len(rlons_temp) == 0 or len(rlats_temp) == 0:
                continue

            # Calculate deposit velocities
            deposit_velocity_components_temp = model.get_point_velocities(rlons_temp, rlats_temp, avg_time)
            deposit_velocity_magnitude_temp = np.hypot(deposit_velocity_components_temp[:, 0], deposit_velocity_components_temp[:, 1])
            deposit_velocity_values.append(deposit_velocity_magnitude_temp)

            # Convert deposit lon/lat to xyz
            deposit_xyz = np.c_[gplately.tools.lonlat2xyz(rlons_temp, rlats_temp)]

            # Find the nearest subduction zone segment for each deposit
            d2sz, sz_neighbours_temp = tree_sz.query(deposit_xyz)

            # Calculate subduction properties
            closest_convergence = subduction_convergence[sz_neighbours_temp].mean()
            closest_migration = subduction_migration[sz_neighbours_temp].mean()

            # Find the nearest craton segment for each deposit
            d2craton, craton_neighbours_temp = tree_craton.query(deposit_xyz)

            # Calculate craton velocities and azimuth
            craton_velocity_components_temp = model.get_point_velocities(
                craton_centroids[craton_neighbours_temp, 1], craton_centroids[craton_neighbours_temp, 0], avg_time
            )
            craton_velocity_magnitude_temp = np.hypot(craton_velocity_components_temp[:, 0], craton_velocity_components_temp[:, 1])
            craton_velocity_angle_temp = np.rad2deg(np.arctan2(craton_velocity_components_temp[:, 0], craton_velocity_components_temp[:, 1]))
            craton_velocity_angle_temp[craton_velocity_angle_temp < 0] += 180

            # Calculate obliquity
            sz_obliquity_temp = abs(subduction_norm[sz_neighbours_temp].mean() - craton_velocity_angle_temp.mean()) % 180
            if sz_obliquity_temp > 90:
                sz_obliquity_temp = 180 - sz_obliquity_temp

            # Append values for median calculation
            craton_velocity_values.append(craton_velocity_magnitude_temp)
            craton_velocity_azimuth_values.append(craton_velocity_angle_temp)
            trench_craton_obliquity_values.append(sz_obliquity_temp)
            trench_convergence_values.append(closest_convergence)
            trench_advance_values.append(closest_migration)

        # Calculate and store medians
        if deposit_velocity_values:
            median_dp_ve = np.median(np.concatenate(deposit_velocity_values))
            median_cr_ve = np.median(np.concatenate(craton_velocity_values))
            median_cr_ve_az = np.median(np.concatenate(craton_velocity_azimuth_values))
            median_tr_cr_an = np.median(trench_craton_obliquity_values)
            median_sb_ve = np.median(trench_convergence_values)
            median_tr_adv = np.median(trench_advance_values)
            
            # Assign medians to all deposits with matching age
            for idx in index_deposit_age:
                df.at[idx, 'dp_ve_m'] = median_dp_ve
                df.at[idx, 'cr_ve_m'] = median_cr_ve
                df.at[idx, 'cr_ve_az_m'] = median_cr_ve_az
                df.at[idx, 'tr_cr_an_m'] = median_tr_cr_an
                df.at[idx, 'sb_ve_m'] = median_sb_ve
                df.at[idx, 'tr_adv_m'] = median_tr_adv

        # ==================== GRADIENT VALUES (over 'g' years) ====================
        # Initialize lists for gradient calculations
        deposit_velocity_vals = []
        craton_velocity_vals = []
        craton_velocity_azimuth_vals = []
        trench_craton_obliquity_vals = []
        trench_convergence_vals = []
        trench_advance_vals = []

        # Loop for the g+1 years to calculate gradients
        for year_offset in range(g + 1):
            avg_time = time - year_offset
            gpts = pts_dict[type_name]
            rlons_grad, rlats_grad = gpts.reconstruct(avg_time, return_array=True)

            valid_mask_grad = (rlats_grad >= -90) & (rlats_grad <= 90) & (rlons_grad >= -180) & (rlons_grad <= 180)
            rlons_grad = rlons_grad[valid_mask_grad]
            rlats_grad = rlats_grad[valid_mask_grad]

            if len(rlons_grad) == 0 or len(rlats_grad) == 0:
                continue

            # Calculate deposit velocities
            deposit_velocity_components_grad = model.get_point_velocities(rlons_grad, rlats_grad, avg_time)
            deposit_velocity_magnitude_grad = np.hypot(deposit_velocity_components_grad[:, 0], deposit_velocity_components_grad[:, 1])
            deposit_velocity_vals.append(deposit_velocity_magnitude_grad.mean())

            # Convert deposit lon/lat to xyz coordinates
            deposit_xyz_grad = np.c_[gplately.tools.lonlat2xyz(rlons_grad, rlats_grad)]

            # Find the nearest subduction zone segment
            d2sz_grad, sz_neighbours_grad = tree_sz.query(deposit_xyz_grad)

            # Calculate subduction zone properties
            sz_norm_grad = subduction_norm[sz_neighbours_grad].copy()
            sz_norm_grad[sz_norm_grad > 180] -= 180

            # Find the nearest craton segment
            d2craton_grad, craton_neighbours_grad = tree_craton.query(deposit_xyz_grad)

            # Calculate craton velocities and azimuth
            craton_velocity_components_grad = model.get_point_velocities(
                craton_centroids[craton_neighbours_grad, 1], craton_centroids[craton_neighbours_grad, 0], avg_time
            )
            craton_velocity_magnitude_grad = np.hypot(craton_velocity_components_grad[:, 0], craton_velocity_components_grad[:, 1])
            craton_velocity_angle_grad = np.rad2deg(np.arctan2(craton_velocity_components_grad[:, 0], craton_velocity_components_grad[:, 1]))
            craton_velocity_angle_grad[craton_velocity_angle_grad < 0] += 180

            craton_velocity_vals.append(craton_velocity_magnitude_grad.mean())
            craton_velocity_azimuth_vals.append(craton_velocity_angle_grad.mean())

            # Calculate obliquity
            sz_obliquity_grad = abs(sz_norm_grad.mean() - craton_velocity_angle_grad.mean()) % 180
            if sz_obliquity_grad > 90:
                sz_obliquity_grad = 180 - sz_obliquity_grad

            trench_craton_obliquity_vals.append(sz_obliquity_grad)
            trench_convergence_vals.append(subduction_convergence[sz_neighbours_grad].mean())
            trench_advance_vals.append(subduction_migration[sz_neighbours_grad].mean())

        # Calculate and store gradients
        if len(deposit_velocity_vals) >= g + 1:
            grad_dp_ve = (deposit_velocity_vals[g] - deposit_velocity_vals[0])
            grad_cr_ve = (craton_velocity_vals[g] - craton_velocity_vals[0])
            grad_cr_ve_az = (craton_velocity_azimuth_vals[g] - craton_velocity_azimuth_vals[0])
            grad_tr_cr_an = (trench_craton_obliquity_vals[g] - trench_craton_obliquity_vals[0])
            grad_sb_ve = (trench_convergence_vals[g] - trench_convergence_vals[0])
            grad_tr_adv = (trench_advance_vals[g] - trench_advance_vals[0])
            
            # Assign gradients to all deposits with matching age
            for idx in index_deposit_age:
                df.at[idx, 'dp_ve_g'] = grad_dp_ve
                df.at[idx, 'cr_ve_g'] = grad_cr_ve
                df.at[idx, 'cr_ve_az_c'] = grad_cr_ve_az
                df.at[idx, 'tr_cr_an_c'] = grad_tr_cr_an
                df.at[idx, 'sb_ve_g'] = grad_sb_ve
                df.at[idx, 'tr_adv_g'] = grad_tr_adv

    # Update progress
    gplately.tools.update_progress((t + 1) / len(unique_times))

# ==================== CALCULATE DISTANCE FROM DEPOSITS TO TRENCH (ds_dp_tr) ====================
unique_times_sorted = sorted(list(unique_times))

# Tessellate subduction zones for the last time step
subduction_data_final = model.tessellate_subduction_zones(unique_times_sorted[-1], np.deg2rad(0.5), ignore_warnings=True)
subduction_lon_final = subduction_data_final[:, 0]
subduction_lat_final = subduction_data_final[:, 1]

# Convert lon/lat to xyz coordinates for subduction zones
sx_final, sy_final, sz_final = gplately.tools.lonlat2xyz(subduction_lon_final, subduction_lat_final)
tree_sz_final = cKDTree(np.c_[sx_final, sy_final, sz_final])

# Iterate over each mineral type to calculate ds_dp_tr
for type_name in types:
    df = metal_dict[type_name]
    gpts = pts_dict[type_name]
    rlons_final, rlats_final = gpts.reconstruct(unique_times_sorted[-1], return_array=True)

    # Apply strict validation
    valid_mask_final = (rlats_final >= -90) & (rlats_final <= 90) & (rlons_final >= -180) & (rlons_final <= 180)
    rlons_final = rlons_final[valid_mask_final]
    rlats_final = rlats_final[valid_mask_final]

    if len(rlats_final) == 0 or len(rlons_final) == 0:
        print(f"No valid lat/lon points for {type_name}")
        continue

    # Convert deposit lon/lat to xyz coordinates
    dx_final, dy_final, dz_final = gplately.tools.lonlat2xyz(rlons_final, rlats_final)

    # Find nearest subduction zone segment for each deposit
    d2s_final, sz_neighbours_final = tree_sz_final.query(np.c_[dx_final, dy_final, dz_final])
    d2s_final = d2s_final * 6371  # Convert distance to kilometers

    # Update the DataFrame with computed distances - assign to each row individually
    for i, dist in enumerate(d2s_final):
        if i < len(df):
            df.at[i, 'ds_dp_tr'] = dist

Progress: [####################] 100.0%


## EXPORT TO CSV WITH ALL ORIGINAL SHAPEFILE COLUMNS

In [33]:
# ==================== EXPORT TO CSV WITH ALL ORIGINAL SHAPEFILE COLUMNS ====================
combined_df_list = []

for type_name in types:
    df = metal_dict[type_name].copy()
    
    # Drop the geometry column for CSV export (keep all other columns)
    if 'geometry' in df.columns:
        df_export = df.drop(columns=['geometry'])
    else:
        df_export = df
    
    combined_df_list.append(df_export)

# Concatenate all DataFrames
if combined_df_list:
    final_combined_df = pd.concat(combined_df_list, ignore_index=True)
    
    # Export to CSV
    final_combined_df.to_csv("Deposits_Features.csv", index=False)
    print(f"All features extracted and saved to CSV with {len(final_combined_df.columns)} columns")
    print(f"Total records: {len(final_combined_df)}")
    print(f"\nColumn names: {list(final_combined_df.columns)}")
else:
    print("No data to export.")

All features extracted and saved to CSV with 65 columns
Total records: 8

Column names: ['Type', 'Deposit', 'Country', 'present_lon', 'present_lat', 'Age__Ga_', 'Age_Uncert', 'Age_Type', 'Ore__Mt_', 'Cu_grade__', 'Zn_grade__', 'Pb_grade__', 'Cu__Mt_', 'Zn__Mt_', 'Pb__Mt_', 'TM', 'weight', 'Ore_Tonnag', 'Ni_grade__', 'Ni__Mt_', 'References', 'NEAR_FID4', 'DIST4', 'NEAR_FID3', 'DIST3', 'NEAR_FID34', 'DIST34', 'Field', 'NEAR_FID', 'DIST44', 'NEAR_Cr', 'DIS_Cr', 'NEAR_34_2', 'DIST34_2', 'NEAR_Cr2', 'DIST_Cr2', 'NEA_Cr_Dep', 'DIS_Cr_Dep', 'DIS_Dep_rc', 'PLATEID1', 'age (Ma)', 'craton_dis', 'craton_az', 'craton_len', 'cr_ve', 'dp_ve', 'ds_cr_tr', 'sb_ve', 'tr_adv', 'tr_cr_an', 'lat', 'lon', 'dp_ve_m', 'cr_ve_m', 'cr_ve_az_m', 'tr_cr_an_m', 'sb_ve_m', 'tr_adv_m', 'dp_ve_g', 'cr_ve_g', 'cr_ve_az_c', 'tr_cr_an_c', 'sb_ve_g', 'tr_adv_g', 'ds_dp_tr']
